# Demostración de enrutamiento y aprendizaje multitarea de SRA

Este cuaderno demuestra la mayor fortaleza de SRA: **"cómo se utilizan selectivamente los expertos (Synapses) en el aprendizaje multitarea"**.

Entrenamos simultáneamente un solo modelo en dos tareas conflictivas: `copy` (salida tal cual) y `reverse` (salida en orden inverso).
Después del entrenamiento, visualizamos **cómo el enrutador selecciona rutas (enrutamiento) de Synapse completamente diferentes dependiendo de la tarea**.

## 1. Configuración del entorno

In [ ]:
import sys
if 'google.colab' in sys.modules:
    !git clone https://github.com/JunSuzukiJapan/SynapticRouter.git
    %cd SynapticRouter
    !pip install torch matplotlib seaborn

sys.path.append('.')
sys.path.append('./src')
if 'google.colab' not in sys.modules:
    sys.path.append('..')
    sys.path.append('../src')


## 2. Preparar las bibliotecas y el modelo
Como queremos manejar múltiples tareas, preparamos algunos expertos más (Synapses).

In [ ]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
import seaborn as sns

from src.sra_gpu_models import MoESRAModel
class MoESRAConfig:
    def __init__(self, **kwargs):
        for k, v in kwargs.items():
            setattr(self, k, v)
from src.sra_experiment import make_multitask_batch, make_optimizer, load_balance_loss

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Use 8 experts (Synapses) and pick 2 of them for each token
config = MoESRAConfig(
    vocab_size=30,
    d_model=64,
    n_layers=2,
    n_heads=2,
    num_synapses=8,  # Bump it up a bit for multitask
    k=2,
    max_seq_len=32
)
model = MoESRAModel(config.vocab_size, config.d_model, config.n_layers, config.num_synapses, config.k, syn_hidden=128).to(device)
optimizer = make_optimizer(model, lr=0.005)

## 3. Ejecutar entrenamiento multitarea
Alimentamos lotes mixtos que contienen tareas `copy` y `reverse` y los entrenamos simultáneamente.

In [ ]:
print("Multitask Training started...")
model.train()

epochs = 500
batch_size = 32
tasks = ["copy", "reverse"]

for epoch in range(epochs):
    # Generate batches with mixed tasks (a task token is prepended to each input)
    x, y, batch_tasks = make_multitask_batch(tasks, batch_size, min_len=4, max_len=8, device=device)
    
    optimizer.zero_grad()
    y_in = torch.cat([torch.full((y.size(0), 1), 1, dtype=torch.long, device=device), y[:, :-1]], dim=1)
    outputs, routing_weights, _ = model(x, y_in)
    
    loss = F.cross_entropy(outputs.reshape(-1, config.vocab_size), y.reshape(-1))
    lb_loss = load_balance_loss(routing_weights) * 0.1
    total_loss = loss + lb_loss
    
    total_loss.backward()
    optimizer.step()
    
    if (epoch + 1) % 100 == 0:
        print(f"Epoch {epoch+1}/{epochs} | Loss: {loss.item():.4f} | LB Loss: {lb_loss.item():.4f}")

print("Training finished!")

## 4. Visualice el enrutamiento y compare entre tareas
Alimentamos el mismo modelo con las entradas `copy` y `reverse` por separado, y comparamos qué Synapses selecciona el router en cada caso.

In [ ]:
def get_routing_weights_for_task(task):
    model.eval()
    # When make_multitask_batch is called with a single task, it generates inputs dedicated to that task
    x, y, _ = make_multitask_batch([task], batch_size=1, min_len=8, max_len=8, device=device)
    
    with torch.no_grad():
        y_in = torch.cat([torch.full((y.size(0), 1), 1, dtype=torch.long, device=device), y[:, :-1]], dim=1)
        _, routing_weights, _ = model(x, y_in)
        
    # Return weights at [Layer 0][Sample 0]
    return routing_weights[0][0].cpu().numpy()

weights_copy = get_routing_weights_for_task("copy")
weights_reverse = get_routing_weights_for_task("reverse")

# Draw the heatmaps side by side
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

sns.heatmap(weights_copy, cmap="Blues", ax=axes[0])
axes[0].set_title("Routing for [COPY] Task")
axes[0].set_xlabel("Synapse Index")
axes[0].set_ylabel("Token Index")

sns.heatmap(weights_reverse, cmap="Oranges", ax=axes[1])
axes[1].set_title("Routing for [REVERSE] Task")
axes[1].set_xlabel("Synapse Index")
axes[1].set_ylabel("Token Index")

plt.tight_layout()
plt.show()

print("=== Observation Points ===")
print("Notice that the positions of the highlighted Synapses (X-axis) differ between")
print("the Copy task on the left (blue) and the Reverse task on the right (orange).")
print("This is evidence that the SRA is 'dynamically switching experts depending on the task'!")

## 5. Visualización de enrutamiento animada
Visualizamos cómo cada token se procesa por turno y se enruta a su Synapse (experto) asignado.
Los valores más pequeños de `speed_ms` hacen que la animación sea más rápida; los valores más grandes lo hacen más lento.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML, display

def animate_routing(weights, task_name, speed_ms=50):
    seq_len, num_synapses = weights.shape
    fig, ax = plt.subplots(figsize=(10, 6))

    ax.set_xlim(-2, 12)
    ax.set_ylim(-1, max(seq_len, num_synapses))
    ax.axis('off')

    # Set coordinates for tokens and Synapses (Y-axis runs top to bottom)
    token_y = np.linspace(seq_len-1, 0, seq_len)
    synapse_y = np.linspace(num_synapses-1, 0, num_synapses)

    # Draw the nodes
    ax.scatter([0]*seq_len, token_y, s=300, c='#1f77b4', zorder=4)
    ax.scatter([10]*num_synapses, synapse_y, s=300, c='#ff7f0e', zorder=4)

    for i in range(seq_len):
        ax.text(-0.8, token_y[i], f'Token {i}', ha='right', va='center', fontsize=12)
    for j in range(num_synapses):
        ax.text(10.8, synapse_y[j], f'Synapse {j}', ha='left', va='center', fontsize=12)

    # Number of animation frames per token
    frames_per_token = 20
    total_frames = seq_len * frames_per_token

    # Initialize particles and trails (lines)
    particles, = ax.plot([], [], 'ro', ms=10, zorder=5) # the moving dot
    trails = []
    for _ in range(seq_len * 2): # up to 2x because k=2
        line, = ax.plot([], [], 'r-', alpha=0.6, lw=2, zorder=3)
        trails.append(line)

    # Pre-compute the connection targets
    connections = []
    for i in range(seq_len):
        targets = np.where(weights[i] > 0)[0]
        connections.append(targets)

    def init():
        particles.set_data([], [])
        for line in trails:
            line.set_data([], [])
        return [particles] + trails

    def update(frame):
        token_idx = frame // frames_per_token
        progress = (frame % frames_per_token) / (frames_per_token - 1)
        
        trail_idx = 0
        current_px, current_py = [], []
        
        for i in range(seq_len):
            targets = connections[i]
            for j in targets:
                start_x, start_y = 0, token_y[i]
                end_x, end_y = 10, synapse_y[j]
                
                if i < token_idx:
                    # Already-completed transmission (kept in gray)
                    trails[trail_idx].set_data([start_x, end_x], [start_y, end_y])
                    trails[trail_idx].set_color('lightgray')
                elif i == token_idx:
                    # Currently transmitting (extending in red)
                    cur_x = start_x + (end_x - start_x) * progress
                    cur_y = start_y + (end_y - start_y) * progress
                    trails[trail_idx].set_data([start_x, cur_x], [start_y, cur_y])
                    trails[trail_idx].set_color('red')
                    current_px.append(cur_x)
                    current_py.append(cur_y)
                else:
                    # Not transmitting yet
                    trails[trail_idx].set_data([], [])
                trail_idx += 1
                
        particles.set_data(current_px, current_py)
        ax.set_title(f'Routing for [{task_name}] Task - Processing Token {token_idx}', fontsize=14)
        return [particles] + trails

    # Build the animation (the `interval` argument is the speed parameter)
    ani = FuncAnimation(fig, update, frames=total_frames, init_func=init, blit=True, interval=speed_ms)
    plt.close()
    return HTML(ani.to_jshtml())

# You can tune the speed (speed_ms) here (default 40ms)
print("Generating animation for the Copy task...")
display(animate_routing(weights_copy, "COPY", speed_ms=40))

# To see the animation for the Reverse task, uncomment the lines below
# print("Generating animation for the Reverse task...")
# display(animate_routing(weights_reverse, "REVERSE", speed_ms=40))